# Gulf of Mexico Ocean Currents

This notebook demonstrates OTP-FM for modeling ocean currents in the Gulf of Mexico, 
based on simulated particle trajectories along observed current velocities around a vortex [1, 2].

[1] https://www.hycom.org/data/gomb0pt01/gom-reanalysis \
[2] Shen, Y., Berlinghieri, R., and Broderick, T. *Multi-marginal Schrodinger bridges with iterative reference refinement.* AISTATS 2025

In [ ]:
import torch
import numpy as np
from collections import OrderedDict
from pathlib import Path

# Import OTP-FM
from otpfm import OTPFM
from otpfm.potentials import W2InfPotential

# Import experiment utilities
from experiments.gulfofmexico.data import load_gom_data, create_gom_dataloaders
from experiments.gulfofmexico import GoMTrainer, plotting

## 1. Load Data

The Gulf of Mexico data will be downloaded automatically.

In [ ]:
# Load data
data = load_gom_data(
    data_dir=Path("data/gom"),
    normalize=True,
    ot_coupling=True,
)

print(f"Number of time points: {len(data['marginals'])}")
print(f"Training times: {data['train_times']}")
print(f"Holdout times: {data['holdout_times']}")
for t, m in data['marginals'].items():
    print(f"  Time {t}: {len(m)} samples")

## 2. Visualize Data

In [ ]:
_ = plotting.plot_scatter(data['marginals'], title="GoM", show=True)

## 3. Create Model and Train

In [ ]:
# Create dataloaders
train_loader, val_loader = create_gom_dataloaders(
    marginals=data['marginals_list'],
    batch_size=64,
    holdout_times=data['holdout_times'],
    ot_alignments=data['ot_alignments'],
)

# Define intermediate times
train_times = data['train_times']
tks = [(t - min(train_times)) / (max(train_times) - min(train_times)) 
       for t in train_times[1:-1]]

# Create potentials
potentials = OrderedDict()
for tk in tks:
    potentials[tk] = W2InfPotential(
        tk=tk,
        strength=100.0,
        lambda_type='gaussian',
        width=0.2,
    )

# Create model (2D data)
model = OTPFM(
    d=2,
    tks=tks,
    potentials=potentials,
    flownet_args={
        'hidden_dim': 128,
        'num_hidden_layers': 3,
    }
)

print(f"Intermediate times: {tks}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Train with GoMTrainer
trainer = GoMTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    save_dir=Path("runs/gom_demo"),
    lr=1e-3,
    epochs=100,
    potentials=potentials,
    marginals=data['marginals'],
    train_times=data['train_times'],
    holdout_times=data['holdout_times'],
    scaler=data['scaler'],
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

trainer.train()

## 4. Visualize Learned Trajectories

In [ ]:
# Sample trajectories
model.eval()
source_time = min(data['train_times'])
x0 = data['marginals'][source_time].to(trainer.device)

with torch.no_grad():
    trajectories, t_eval = model.sample(x0, n_steps=25, ema=True)

# Use built-in plotting
plotting.plot_trajectories(
    trajectories=trajectories,
    time_points=t_eval.numpy(),
    ground_truth_marginals=data['marginals'],
    num_trajectories=100,
    title="Gulf of Mexico Ocean Currents",
    show=True,
)

## Reproduce Paper Results

```bash
python experiments/train.py --dataset gulfofmexico --potential {W2, W2Inf, KL, MMD}
```